In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
files = os.listdir(
    "/content/drive/MyDrive/Colab Notebooks/output/statistics"
)
files

['sampled_all_systems.csv',
 'concat_flaml.csv',
 'concat_ag.csv',
 'concat_rf.csv',
 'hpo_all_systems.csv',
 'result_equal_monotonic_rf_equal.csv',
 'result_monotonic_rf_equal.csv',
 'result_equal_monotonic_flaml_equal.csv',
 'result_monotonic_flaml_equal.csv',
 'result_equal_monotonic_ag_equal.csv',
 'result_monotonic_ag_equal.csv',
 'result_monotonic_llama.csv',
 'result_monotonic_mistral.csv',
 'llama.csv',
 'mistral.csv']

# AutoGluon

In [ ]:
system = "ag"
system_map = {
    "rf": "Random Forest with Random Search",
    "ag": "AutoGluon",
    "flaml": "FLAML",
}

## HPO

In [ ]:
hpo = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/output/statistics/hpo_all_systems.csv",
    sep=";"
)
hpo = hpo[hpo["automl_system"] == system_map[system]]
hpo["are_concat_pred_equal"] = hpo.apply(
    lambda row: row["trained_dataset"] == row["predicted_dataset"],
    axis=1
)
equal_hpo = hpo[hpo["are_concat_pred_equal"]]
diff_hpo = hpo[~(hpo["are_concat_pred_equal"])]

In [ ]:
# Looking at one HPO equal dataset example
equal_hpo.query("trained_dataset == 'ua_01'")

,trained_dataset,predicted_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels,are_concat_pred_equal
323,ua_01,ua_01,AutoGluon,gmm_random,100,[],[],False,0.55838,0.658105,"['3', '10', '1', '7', '5']","['3', '10', '1', '7', '5']",True


In [ ]:
# Looking at diff HPO equal dataset example
diff_hpo.query("trained_dataset == 'ua_01'")

,trained_dataset,predicted_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels,are_concat_pred_equal
322,ua_01,ua_03,AutoGluon,gmm_random,100,[],[],False,0.666016,0.630244,"['10', '3', '1', '7', '5']","['3', '10', '1', '7', '5']",False


## Concat

In [ ]:
automl_concat_df = pd.read_csv(
    f"/content/drive/MyDrive/Colab Notebooks/output/statistics/concat_{system}.csv",
    sep=";"
)

adapt_mask = automl_concat_df["concatenated_dataset"] == automl_concat_df["predicted_dataset"]
aug_mask = automl_concat_df["trained_dataset"] == automl_concat_df["predicted_dataset"]

equal_concat_df = automl_concat_df.loc[adapt_mask & (~aug_mask)]
diff_concat_df = automl_concat_df.loc[(~adapt_mask) & (~aug_mask)]

In [ ]:
# Looking at one equal concat dataset example
equal_concat_df.query("trained_dataset == 'ua_01'")

,trained_dataset,predicted_dataset,concatenated_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
1651,ua_01,ua_03,ua_03,AutoGluon,gmm_random,25,['3'],[],False,0.666016,0.661789,"['3', '5', '10', '1', '7']","['5', '1', '10', '7']"
1656,ua_01,ua_03,ua_03,AutoGluon,gmm_random,50,[],[],False,0.666016,0.668618,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
1661,ua_01,ua_03,ua_03,AutoGluon,gmm_random,75,['3'],[],False,0.666016,0.659837,"['3', '5', '10', '1', '7']","['5', '1', '10', '7']"
1667,ua_01,ua_04,ua_04,AutoGluon,gmm_random,25,[],[],False,0.200040,0.483397,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
1672,ua_01,ua_04,ua_04,AutoGluon,gmm_random,50,[],[],False,0.200040,0.483597,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
1677,ua_01,ua_04,ua_04,AutoGluon,gmm_random,75,[],[],False,0.200040,0.488398,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
1683,ua_01,ua_05,ua_05,AutoGluon,gmm_random,25,['3'],[],False,0.776119,0.763682,"['3', '5', '10', '1', '7']","['7', '5', '10', '1']"
1688,ua_01,ua_05,ua_05,AutoGluon,gmm_random,50,['3'],[],False,0.776119,0.763682,"['3', '5', '10', '1', '7']","['7', '5', '10', '1']"
1693,ua_01,ua_05,ua_05,AutoGluon,gmm_random,75,['3'],[],False,0.776119,0.783582,"['3', '5', '10', '1', '7']","['7', '5', '10', '1']"
1699,ua_01,ua_06,ua_06,AutoGluon,gmm_random,25,[],[],False,0.442095,0.565875,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"


In [ ]:
# Looking at one diff concat dataset example
diff_concat_df.query("trained_dataset == 'ua_01'")

,trained_dataset,predicted_dataset,concatenated_dataset,automl_system,method,percentage,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
1652,ua_01,ua_04,ua_03,AutoGluon,gmm_random,25,[],[],False,0.200040,0.368074,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
1653,ua_01,ua_05,ua_03,AutoGluon,gmm_random,25,['3'],[],False,0.776119,0.686567,"['3', '5', '10', '1', '7']","['7', '5', '10', '1']"
1654,ua_01,ua_06,ua_03,AutoGluon,gmm_random,25,['3'],[],False,0.442095,0.480481,"['3', '5', '10', '1', '7']","['7', '5', '10', '1']"
1657,ua_01,ua_04,ua_03,AutoGluon,gmm_random,50,[],[],False,0.200040,0.362973,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
1658,ua_01,ua_05,ua_03,AutoGluon,gmm_random,50,['3'],[],False,0.776119,0.686567,"['3', '5', '10', '1', '7']","['7', '5', '10', '1']"
1659,ua_01,ua_06,ua_03,AutoGluon,gmm_random,50,"['3', '5', '7']",[],False,0.442095,0.461451,"['3', '5', '10', '1', '7']","['10', '1']"
1662,ua_01,ua_04,ua_03,AutoGluon,gmm_random,75,[],[],False,0.200040,0.351270,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
1663,ua_01,ua_05,ua_03,AutoGluon,gmm_random,75,['3'],[],False,0.776119,0.671642,"['3', '5', '10', '1', '7']","['7', '5', '10', '1']"
1664,ua_01,ua_06,ua_03,AutoGluon,gmm_random,75,"['3', '5', '7']",[],False,0.442095,0.468445,"['3', '5', '10', '1', '7']","['10', '1']"
1666,ua_01,ua_03,ua_04,AutoGluon,gmm_random,25,[],[],False,0.666016,0.513171,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"


## Equal

### 25, 50, 75

In [ ]:
grouped_equal_concat_df = (
    equal_concat_df.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method",
            "percentage"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_equal_concat_df = grouped_equal_concat_df.reset_index()
grouped_equal_concat_df

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score
0,ed_01,AutoGluon,gmm_random,25,0.132780
1,ed_01,AutoGluon,gmm_random,50,0.302905
2,ed_01,AutoGluon,gmm_random,75,0.294606
3,ed_02,AutoGluon,gmm_random,25,0.667767
4,ed_02,AutoGluon,gmm_random,50,0.638624
...,...,...,...,...,...
100,ua_05,AutoGluon,gmm_random,50,0.766667
101,ua_05,AutoGluon,gmm_random,75,0.787065
102,ua_06,AutoGluon,gmm_random,25,0.571080
103,ua_06,AutoGluon,gmm_random,50,0.576350


### 0%

In [ ]:
grouped_zero_pct_hpo = (
    diff_hpo.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_zero_pct_hpo = grouped_zero_pct_hpo.reset_index()
grouped_zero_pct_hpo["percentage"] = 0
grouped_zero_pct_hpo

,predicted_dataset,automl_system,method,hold_out_f1_score,percentage
0,ed_01,AutoGluon,gmm_random,0.020747,0
1,fn_02,AutoGluon,gmm_random,0.344444,0
2,fn_06,AutoGluon,gmm_random,0.362869,0
3,fn_07,AutoGluon,gmm_random,0.394886,0
4,fn_09,AutoGluon,gmm_random,0.285834,0
5,fn_10,AutoGluon,gmm_random,0.366667,0
6,hs_01,AutoGluon,gmm_random,0.450348,0
7,hs_02,AutoGluon,gmm_random,0.314387,0
8,hs_03,AutoGluon,gmm_random,0.584734,0
9,hs_04,AutoGluon,gmm_random,0.830148,0


### 100%

In [ ]:
# There is only one point per dataset since trained = predicted, but we are still grouping by and determining the mean for consistency
grouped_100_pct_hpo = (
    equal_hpo.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_100_pct_hpo = grouped_100_pct_hpo.reset_index()
grouped_100_pct_hpo["percentage"] = 100
grouped_100_pct_hpo

,predicted_dataset,automl_system,method,hold_out_f1_score,percentage
0,ed_01,AutoGluon,gmm_random,0.468880,100
1,ed_02,AutoGluon,gmm_random,0.766363,100
2,ed_03,AutoGluon,gmm_random,0.470735,100
3,ed_04,AutoGluon,gmm_random,0.925421,100
4,ed_05,AutoGluon,gmm_random,0.375000,100
5,fn_01,AutoGluon,gmm_random,0.267238,100
6,fn_02,AutoGluon,gmm_random,0.500000,100
7,fn_03,AutoGluon,gmm_random,0.267678,100
8,fn_04,AutoGluon,gmm_random,0.559633,100
9,fn_05,AutoGluon,gmm_random,0.538171,100


### Result

In [ ]:
result_equal = (
    pd.concat(
        [
            grouped_equal_concat_df,
            grouped_zero_pct_hpo,
            grouped_100_pct_hpo
        ],
        axis=0
    )
    .sort_values(
        [
            "predicted_dataset",
            "automl_system",
            "method",
            "percentage"
        ]
    )
)
monotonic_equal = (
    result_equal.sort_values(
        [
            "predicted_dataset",
            "automl_system",
            "method",
            "percentage"
        ]
    )
    .groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ],
        as_index=False,
        observed=True
    )
    ["hold_out_f1_score"]
    .apply(lambda x: x.is_monotonic_increasing)
    .rename(columns={"hold_out_f1_score": "is_monotonic_increasing"})
    [["predicted_dataset", "automl_system", "method", "is_monotonic_increasing"]]
)
result_equal = (
    result_equal.merge(
        monotonic_equal,
        on=["predicted_dataset", "automl_system", "method"],
    )
    .sort_values(["predicted_dataset", "automl_system", "method", "percentage"])
)

value_counts = result_equal["predicted_dataset"].value_counts()

assert value_counts[(value_counts > 1) & (value_counts < 4)].empty, "There should not be any dataset with 2-3 data points"

isolated = value_counts[value_counts == 1].index
assert ~grouped_equal_concat_df["predicted_dataset"].isin(isolated).any(), "There should not be any isolated datasets in concatenated datasets"
assert ~grouped_zero_pct_hpo["predicted_dataset"].isin(isolated).any(), "There should not be any isolated datasets in HPO's 0 pct"
assert (set(isolated) - set(grouped_100_pct_hpo["predicted_dataset"].unique())) == set()

result_equal = result_equal[~result_equal["predicted_dataset"].isin(isolated)]
result_equal.to_csv(
    f"/content/drive/MyDrive/Colab Notebooks/output/statistics/result_equal_monotonic_{system}_equal.csv",
    sep=";"
)
result_equal

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score,is_monotonic_increasing
0,ed_01,AutoGluon,gmm_random,0,0.020747,False
1,ed_01,AutoGluon,gmm_random,25,0.132780,False
2,ed_01,AutoGluon,gmm_random,50,0.302905,False
3,ed_01,AutoGluon,gmm_random,75,0.294606,False
4,ed_01,AutoGluon,gmm_random,100,0.468880,False
...,...,...,...,...,...,...
182,ua_06,AutoGluon,gmm_random,0,0.438842,True
183,ua_06,AutoGluon,gmm_random,25,0.571080,True
184,ua_06,AutoGluon,gmm_random,50,0.576350,True
185,ua_06,AutoGluon,gmm_random,75,0.585817,True


## Diff

### 25, 50, 75

In [ ]:
grouped_diff_concat_df = (
    diff_concat_df.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method",
            "percentage"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_diff_concat_df = grouped_diff_concat_df.reset_index()
grouped_diff_concat_df

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score
0,ed_01,AutoGluon,gmm_random,25,0.012448
1,ed_01,AutoGluon,gmm_random,50,0.020747
2,ed_01,AutoGluon,gmm_random,75,0.037344
3,ed_02,AutoGluon,gmm_random,25,0.107823
4,ed_02,AutoGluon,gmm_random,50,0.147415
...,...,...,...,...,...
70,ua_05,AutoGluon,gmm_random,50,0.627488
71,ua_05,AutoGluon,gmm_random,75,0.616449
72,ua_06,AutoGluon,gmm_random,25,0.467510
73,ua_06,AutoGluon,gmm_random,50,0.465355


### 0%

In [ ]:
grouped_zero_pct_hpo = (
      diff_hpo.groupby(
          [
              "predicted_dataset",
              "automl_system",
              "method"
          ]
      )
      .agg(
          {
              "hold_out_f1_score": "mean"
          }
      )
)
grouped_zero_pct_hpo = grouped_zero_pct_hpo.reset_index()
grouped_zero_pct_hpo["percentage"] = 0
grouped_zero_pct_hpo

,predicted_dataset,automl_system,method,hold_out_f1_score,percentage
0,ed_01,AutoGluon,gmm_random,0.020747,0
1,fn_02,AutoGluon,gmm_random,0.344444,0
2,fn_06,AutoGluon,gmm_random,0.362869,0
3,fn_07,AutoGluon,gmm_random,0.394886,0
4,fn_09,AutoGluon,gmm_random,0.285834,0
5,fn_10,AutoGluon,gmm_random,0.366667,0
6,hs_01,AutoGluon,gmm_random,0.450348,0
7,hs_02,AutoGluon,gmm_random,0.314387,0
8,hs_03,AutoGluon,gmm_random,0.584734,0
9,hs_04,AutoGluon,gmm_random,0.830148,0


### 100%

In [ ]:
# There is only one point per dataset since trained = predicted, but we are still grouping by and determining the mean for consistency
grouped_100_pct_hpo = (
    equal_hpo.groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_100_pct_hpo = grouped_100_pct_hpo.reset_index()
grouped_100_pct_hpo["percentage"] = 100
grouped_100_pct_hpo

,predicted_dataset,automl_system,method,hold_out_f1_score,percentage
0,ed_01,AutoGluon,gmm_random,0.468880,100
1,ed_02,AutoGluon,gmm_random,0.766363,100
2,ed_03,AutoGluon,gmm_random,0.470735,100
3,ed_04,AutoGluon,gmm_random,0.925421,100
4,ed_05,AutoGluon,gmm_random,0.375000,100
5,fn_01,AutoGluon,gmm_random,0.267238,100
6,fn_02,AutoGluon,gmm_random,0.500000,100
7,fn_03,AutoGluon,gmm_random,0.267678,100
8,fn_04,AutoGluon,gmm_random,0.559633,100
9,fn_05,AutoGluon,gmm_random,0.538171,100


### Result

In [ ]:
result_diff = (
    pd.concat(
        [
            grouped_diff_concat_df,
            grouped_zero_pct_hpo,
            grouped_100_pct_hpo
        ],
        axis=0
    )
    .sort_values(["predicted_dataset", "automl_system", "method", "percentage"])
)
monotonic_diff = (
    result_diff.sort_values(["predicted_dataset", "automl_system", "method", "percentage"])
    .groupby(
        [
            "predicted_dataset",
            "automl_system",
            "method"
        ],
        as_index=False, observed=True
    )
    ["hold_out_f1_score"]
    .apply(lambda x: x.is_monotonic_increasing)
    .rename(columns={"hold_out_f1_score": "is_monotonic_increasing"})
    [["predicted_dataset", "automl_system", "method", "is_monotonic_increasing"]]
)
result_diff = (
    result_diff.merge(
        monotonic_diff,
        on=["predicted_dataset", "automl_system", "method"],
    )
    .sort_values(["predicted_dataset", "automl_system", "method", "percentage"])
)

value_counts = result_diff["predicted_dataset"].value_counts()

# assert value_counts[(value_counts > 1) & (value_counts < 4)].empty, "There should not be any dataset with 2-3 data points"

isolated = value_counts[value_counts == 1].index
assert (~grouped_equal_concat_df["predicted_dataset"].isin(isolated)).any(), "There should not be any isolated datasets in concatenated datasets"
assert (~grouped_zero_pct_hpo["predicted_dataset"].isin(isolated)).any(), "There should not be any isolated datasets in HPO's 0 pct"
assert (set(isolated) - set(grouped_100_pct_hpo["predicted_dataset"].unique())) == set()

result_diff = result_diff[~result_diff["predicted_dataset"].isin(isolated)]
result_diff.to_csv(
    f"/content/drive/MyDrive/Colab Notebooks/output/statistics/result_monotonic_{system}_equal.csv",
    sep=";"
)
result_diff

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score,is_monotonic_increasing
0,ed_01,AutoGluon,gmm_random,0,0.020747,False
1,ed_01,AutoGluon,gmm_random,25,0.012448,False
2,ed_01,AutoGluon,gmm_random,50,0.020747,False
3,ed_01,AutoGluon,gmm_random,75,0.037344,False
4,ed_01,AutoGluon,gmm_random,100,0.468880,False
...,...,...,...,...,...,...
152,ua_06,AutoGluon,gmm_random,0,0.438842,False
153,ua_06,AutoGluon,gmm_random,25,0.467510,False
154,ua_06,AutoGluon,gmm_random,50,0.465355,False
155,ua_06,AutoGluon,gmm_random,75,0.467245,False


# Analysis Examples

Some examples of interesting analysis below:


## Monotonic Analysis

### Equal count

In [ ]:
is_monotonic_increasing_equal = result_equal["is_monotonic_increasing"]

monotonic_increasing_equal = result_equal.loc[is_monotonic_increasing_equal]
non_monotonic_increasing_equal = result_equal.loc[~is_monotonic_increasing_equal]

print(f"Number EQUAL Monotonic increasing: {len(monotonic_increasing_equal.drop_duplicates('predicted_dataset'))}")
print(f"{monotonic_increasing_equal['predicted_dataset'].drop_duplicates().to_list()}")

print(f"Number EQUAL Non-monotonic increasing: {len(non_monotonic_increasing_equal.drop_duplicates('predicted_dataset'))}")
print(f"{non_monotonic_increasing_equal['predicted_dataset'].drop_duplicates().to_list()}")

Number EQUAL Monotonic increasing: 19
['fn_07', 'fn_10', 'hs_02', 'hs_03', 'hs_04', 'hs_08', 'hs_09', 'hs_11', 'hs_12', 'hs_13', 'hs_14', 'pc_01', 'pc_03', 'sd_03', 'sd_04', 'ua_01', 'ua_04', 'ua_05', 'ua_06']
Number EQUAL Non-monotonic increasing: 16
['ed_01', 'ed_02', 'fn_02', 'fn_05', 'fn_06', 'fn_09', 'hs_01', 'hs_06', 'hs_10', 'pc_02', 'pc_04', 'pc_05', 'pc_06', 'pc_07', 'pc_08', 'ua_03']


### Diff count

In [ ]:
is_monotonic_increasing_diff = result_diff["is_monotonic_increasing"]

monotonic_increasing_diff = result_diff.loc[is_monotonic_increasing_diff]
non_monotonic_increasing_diff = result_diff.loc[~is_monotonic_increasing_diff]

print(f"Number diff Monotonic increasing: {len(monotonic_increasing_diff.drop_duplicates('predicted_dataset'))}")
print(f"{monotonic_increasing_diff['predicted_dataset'].drop_duplicates().to_list()}")

print(f"Number diff Non-monotonic increasing: {len(non_monotonic_increasing_diff.drop_duplicates('predicted_dataset'))}")
print(f"{non_monotonic_increasing_diff['predicted_dataset'].drop_duplicates().to_list()}")

Number diff Monotonic increasing: 13
['fn_02', 'fn_07', 'fn_09', 'fn_10', 'hs_01', 'hs_04', 'hs_08', 'hs_11', 'hs_14', 'sd_03', 'sd_04', 'ua_01', 'ua_04']
Number diff Non-monotonic increasing: 21
['ed_01', 'ed_02', 'fn_06', 'hs_02', 'hs_03', 'hs_06', 'hs_09', 'hs_10', 'hs_12', 'hs_13', 'pc_01', 'pc_02', 'pc_03', 'pc_04', 'pc_05', 'pc_06', 'pc_07', 'pc_08', 'ua_03', 'ua_05', 'ua_06']


### Non-monotonicic equal f1_score difference

In [ ]:
(
    pd.concat(
        [
            non_monotonic_increasing_equal,
            (
                non_monotonic_increasing_equal[["hold_out_f1_score", "predicted_dataset"]]
                .shift(1)
                .rename(
                    columns={
                        "hold_out_f1_score": "hold_out_f1_score_next",
                        "predicted_dataset": "predicted_dataset_next"
                    }
                )
            )
        ],
        axis=1
    )
    .query("predicted_dataset == predicted_dataset_next")
    .assign(diff=lambda x: x["hold_out_f1_score"] - x["hold_out_f1_score_next"])
    [["predicted_dataset", "automl_system", "method", "percentage", "hold_out_f1_score", "diff"]]
    # TODO: Remove lines below
    # TODO: Analyze cases where diff is in 100
    .query("diff < 0")
    .sort_values("diff", ignore_index=True)
    # .query("predicted_dataset == 'pc_02'")
)

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score,diff
0,pc_02,AutoGluon,gmm_random,100,0.694051,-0.095609
1,fn_06,AutoGluon,gmm_random,100,0.459916,-0.081857
2,fn_05,AutoGluon,gmm_random,50,0.441110,-0.078434
3,fn_02,AutoGluon,gmm_random,100,0.500000,-0.029167
4,ed_02,AutoGluon,gmm_random,50,0.638624,-0.029143
5,pc_08,AutoGluon,gmm_random,100,0.483202,-0.020967
6,pc_04,AutoGluon,gmm_random,75,0.846674,-0.020400
7,pc_06,AutoGluon,gmm_random,100,0.881172,-0.017389
8,hs_06,AutoGluon,gmm_random,50,0.582136,-0.015129
9,pc_05,AutoGluon,gmm_random,100,0.573810,-0.010935


### Non-monotonic diff f1_score difference

In [ ]:
(
    pd.concat(
        [
            non_monotonic_increasing_diff,
            (
                non_monotonic_increasing_diff[["hold_out_f1_score", "predicted_dataset"]]
                .shift(1)
                .rename(
                    columns={
                        "hold_out_f1_score": "hold_out_f1_score_next",
                        "predicted_dataset": "predicted_dataset_next"
                    }
                )
            )
        ],
        axis=1
    )
    .query("predicted_dataset == predicted_dataset_next")
    .assign(diff=lambda x: x["hold_out_f1_score"] - x["hold_out_f1_score_next"])
    [["predicted_dataset", "automl_system", "method", "percentage", "hold_out_f1_score", "diff"]]
    .query("diff < 0")
    .sort_values("diff", ignore_index=True)
)

,predicted_dataset,automl_system,method,percentage,hold_out_f1_score,diff
0,fn_06,AutoGluon,gmm_random,50,0.390295,-0.080169
1,fn_06,AutoGluon,gmm_random,100,0.459916,-0.053797
2,ua_05,AutoGluon,gmm_random,25,0.611940,-0.046020
3,ua_03,AutoGluon,gmm_random,25,0.570081,-0.038618
4,hs_02,AutoGluon,gmm_random,50,0.384569,-0.032537
5,pc_08,AutoGluon,gmm_random,25,0.344645,-0.026811
6,ed_02,AutoGluon,gmm_random,75,0.121949,-0.025466
7,pc_08,AutoGluon,gmm_random,75,0.324179,-0.022836
8,pc_03,AutoGluon,gmm_random,50,0.701477,-0.020723
9,pc_06,AutoGluon,gmm_random,50,0.725788,-0.018079


### 0-100 diff (equal)

In [ ]:
da_analysis_equal = result_equal[result_equal["percentage"].isin([0, 100])] # or result_equal since it has the same effect
da_analysis_equal = da_analysis_equal.merge(da_analysis_equal, on=["predicted_dataset", "automl_system", "method"])
da_analysis_equal = da_analysis_equal[
    (da_analysis_equal["percentage_x"] != da_analysis_equal["percentage_y"])
]
da_analysis_equal = da_analysis_equal.drop_duplicates(["predicted_dataset", "automl_system", "method"])
da_analysis_equal["diff"] = da_analysis_equal["hold_out_f1_score_y"] - da_analysis_equal["hold_out_f1_score_x"]
da_analysis_equal.sort_values("diff", ignore_index=True)

,predicted_dataset,automl_system,method,percentage_x,hold_out_f1_score_x,is_monotonic_increasing_x,percentage_y,hold_out_f1_score_y,is_monotonic_increasing_y,diff
0,hs_06,AutoGluon,gmm_random,0,0.588944,False,100,0.579677,False,-0.009266
1,ua_03,AutoGluon,gmm_random,0,0.608699,False,100,0.666667,False,0.057967
2,hs_04,AutoGluon,gmm_random,0,0.830148,True,100,0.925574,True,0.095426
3,fn_06,AutoGluon,gmm_random,0,0.362869,False,100,0.459916,False,0.097046
4,hs_10,AutoGluon,gmm_random,0,0.602222,False,100,0.700000,False,0.097778
5,pc_02,AutoGluon,gmm_random,0,0.595963,False,100,0.694051,False,0.098088
6,ua_01,AutoGluon,gmm_random,0,0.552515,True,100,0.658105,True,0.105590
7,pc_05,AutoGluon,gmm_random,0,0.467868,False,100,0.573810,False,0.105942
8,hs_09,AutoGluon,gmm_random,0,0.842219,True,100,0.949919,True,0.107701
9,pc_08,AutoGluon,gmm_random,0,0.371456,False,100,0.483202,False,0.111746


### 0-100 diff (diff)

In [ ]:
da_analysis_diff = result_diff[result_diff["percentage"].isin([0, 100])] # or result_equal since it has the same effect
da_analysis_diff = da_analysis_diff.merge(da_analysis_diff, on=["predicted_dataset", "automl_system", "method"])
da_analysis_diff = da_analysis_diff[
    (da_analysis_diff["percentage_x"] != da_analysis_diff["percentage_y"])
]
da_analysis_diff = da_analysis_diff.drop_duplicates(["predicted_dataset", "automl_system", "method"])
da_analysis_diff["diff"] = da_analysis_diff["hold_out_f1_score_y"] - da_analysis_diff["hold_out_f1_score_x"]
da_analysis_diff.sort_values("diff", ignore_index=True)

,predicted_dataset,automl_system,method,percentage_x,hold_out_f1_score_x,is_monotonic_increasing_x,percentage_y,hold_out_f1_score_y,is_monotonic_increasing_y,diff
0,hs_06,AutoGluon,gmm_random,0,0.588944,False,100,0.579677,False,-0.009266
1,ua_03,AutoGluon,gmm_random,0,0.608699,False,100,0.666667,False,0.057967
2,hs_04,AutoGluon,gmm_random,0,0.830148,True,100,0.925574,True,0.095426
3,fn_06,AutoGluon,gmm_random,0,0.362869,False,100,0.459916,False,0.097046
4,hs_10,AutoGluon,gmm_random,0,0.602222,False,100,0.700000,False,0.097778
5,pc_02,AutoGluon,gmm_random,0,0.595963,False,100,0.694051,False,0.098088
6,ua_01,AutoGluon,gmm_random,0,0.552515,True,100,0.658105,True,0.105590
7,pc_05,AutoGluon,gmm_random,0,0.467868,False,100,0.573810,False,0.105942
8,hs_09,AutoGluon,gmm_random,0,0.842219,False,100,0.949919,False,0.107701
9,pc_08,AutoGluon,gmm_random,0,0.371456,False,100,0.483202,False,0.111746
